# Score a BENDL ensemble

<style>
blockquote:has(.notebook-admonition-title) {
  --notebook-admonition-color: var(--color-admonition-title--note, #087fc7);
  --notebook-admonition-title-background:
    var(--color-admonition-title-background--note, rgba(8, 127, 199, 0.18));
  background: var(--color-admonition-background, transparent);
  border: 0;
  border-left: 0.2rem solid var(--notebook-admonition-color);
  border-radius: 0.2rem;
  box-shadow: 0 0.2rem 0.5rem rgba(0, 0, 0, 0.05), 0 0 0.0625rem rgba(0, 0, 0, 0.1);
  font-size: var(--admonition-font-size, 0.8125rem);
  margin: 1rem auto;
  overflow: hidden;
  padding: 0 0.5rem 0.5rem;
}
blockquote p:has(> .notebook-admonition-title) {
  background: var(--notebook-admonition-title-background);
  font-size: var(--admonition-title-font-size, 0.8125rem);
  font-weight: 500;
  line-height: 1.3;
  margin: 0 -0.5rem 0.5rem;
  padding: 0.4rem 0.5rem 0.4rem 2rem;
  position: relative;
}
blockquote p:has(> .notebook-admonition-title)::before {
  color: var(--notebook-admonition-color);
  content: "✎";
  left: 0.65rem;
  position: absolute;
}
.notebook-admonition-title {
  font-weight: inherit;
}
table:not(.dataframe) {
  border: 1px solid var(--docs-hairline, rgba(128, 128, 128, 0.35));
  border-collapse: collapse;
}
table:not(.dataframe) th,
table:not(.dataframe) td {
  border: 1px solid var(--docs-hairline, rgba(128, 128, 128, 0.35));
}
</style>

<div style="text-align: center;"><a class="sd-sphinx-override sd-btn sd-text-wrap sd-btn-primary reference external" href="https://www.dropbox.com/scl/fo/s22x9phl0hldiakn8nbuz/ABKfxHBaak5ra3eBGkNFWMM?rlkey=igpo7qi07oz5tfgjki317o79t&amp;st=gcxkicnc&amp;dl=1">Download tutorial data</a></div>

This guide uses one committed Colorado VTD BENDL fixture (`data/co_vtd_scoring_10000.bendl`). The 
bundle contains:

- a 10,000-step, ReCom chain;
- its 3,158-node dual graph and run metadata;
- a GeoParquet asset with projected VTD geometry, population, and election columns; and
- fixture provenance, including the two small connectivity repairs made to the VTD seed.

In [ ]:
import tempfile
from pathlib import Path

import numpy as np
import pandas as pd

import gerrytools.scoring as gs
from gerrytools.ben import BendlDecoder, read_geoparquet_asset

## Open the BENDL resources

`BendlDecoder` verifies the checksums, reads the embedded graph, and exposes arbitrary assets.
`read_geoparquet_asset()` reads an embedded GeoParquet directly into a GeoDataFrame. For an
ordinary Parquet table, use `read_parquet_asset()` instead. Assignment positions in the BENDL
stream follow the embedded graph's node order, and the `node` column records the same order
explicitly.

In [ ]:
data_dir = Path("data")
bundle_path = data_dir / "co_vtd_scoring_10000.bendl"

bundle = BendlDecoder(bundle_path)
graph = bundle.read_graph()
vtds = read_geoparquet_asset(bundle, "co_vtds_2020.parquet")

In [ ]:
pd.Series(
    {
        "VTDs": len(vtds),
        "graph edges": graph.number_of_edges(),
        "districts": vtds["assignment"].nunique(),
        "chain samples": bundle.count_samples(),
        "CRS": vtds.crs.to_string(),
        "assets": ", ".join(bundle.asset_names()),
    },
    name="Colorado scoring fixture",
)

## Configure an evaluator

The evaluator records borrowed graph and geometry sources when it is configured, then snapshots
only the requested resources during the first evaluation. Later evaluations reuse that immutable
snapshot and the scoring engine. Do not mutate either source after preparation; adding another
metric extends the snapshot only with resources that were not already prepared. The active
GeoDataFrame geometry column is reserved for geometry-backed metrics. This example registers
representative district, plan, region, election, and compactness metrics:

- `Tally` combines all requested numeric columns into one engine pass.
- `PolsbyPopper` demonstrates geometry-backed compactness.
- `CutEdges` demonstrates unweighted counts and shared-perimeter weights.
- The three region statistics use counties as fixed regions.
- `TallyByRegion` produces a county-by-district table with named values.

Each metric owns its optional `result_name`, so aliases work naturally with
`add_metrics(...)`. Compatible metrics still share engine state, but that implementation detail
never changes how results are accessed.

In [ ]:
graph.nodes(data=True)[0]

In [ ]:
evaluator = gs.PlanEvaluator(graph, geometry=vtds, node_id_column="node")
evaluator.add_metrics(
    gs.Tally(
        "total_pop_20",
        "total_vap_20",
        "bvap_20",
        "pres_16_dem",
        "pres_16_rep",
        "pres_20_dem",
        "pres_20_rep",
        "pres_24_dem",
        "pres_24_rep",
        result_name="district_totals",
    ),
    gs.PolsbyPopper(),
    gs.Reock(),
    gs.CutEdges(result_name="cut_edge_count"),
    gs.CutEdges(weight_attr="shared_perim", result_name="cut_edge_perimeter"),
    gs.RegionSplits("county", result_name="county_splits"),
    gs.RegionPieces("county", result_name="county_pieces"),
    gs.RegionParts("county", result_name="county_parts"),
    gs.TallyByRegion(
        "county",
        {"population": "total_pop_20", "BVAP": "bvap_20"},
        include_count=True,
        result_name="county_totals",
    ),
)

## Evaluate one assignment with `lookup`

`lookup(i)` returns one assignment vector without decoding earlier plans. For a one-off score,
call a lowercase function with a GeoDataFrame and either an assignment column or assignment
vector. The function constructs a temporary evaluator and returns a pandas object or scalar.
Geometry scores use the same pattern, for example `gs.polsby_popper(vtds, "assignment")`; use a
persistent evaluator when the statewide geometry will be reused.

In [ ]:
single_assignment = bundle.lookup(0)
direct_population = gs.tally(vtds, single_assignment, columns="total_pop_20")
direct_population_deviations = gs.population_deviations(
    graph,
    single_assignment,
    population_attr="total_pop_20",
)
pd.DataFrame(
    {"population": direct_population, "population deviation": direct_population_deviations}
)

In [ ]:
single = evaluator.evaluate(single_assignment)
single.metrics

In [ ]:
single["district_totals"]

In [ ]:
single["reock"]

In [ ]:
pd.Series(
    {name: single[name] for name in ("county_splits", "county_pieces", "county_parts")},
    name="county metrics",
)

With no graph-attribute names supplied, Polsby-Popper derives its measurements from the aligned
GeoDataFrame geometry. Supplying any graph-attribute name would instead select the embedded
graph and require every compactness measurement to come from its attributes.

In [ ]:
single["polsby_popper"]

## Evaluate selected assignments with `subsample_indices`

`subsample_indices` decodes only the requested zero-based sample indices. Materialize that small
selection before passing it to `evaluate_many`; `sample_ids` then assigns meaningful, unique
labels to the result rows rather than requiring the caller to relabel each table. Every
assignment in a batch must use the same district-label set.

In [ ]:
sample_indices = [0, 100, 1_000, 9_999]
selected_assignments = list(bundle.subsample_indices(sample_indices))
selected = evaluator.evaluate_many(
    selected_assignments,
    sample_ids=sample_indices,
)
selected["cut_edge_count"]

## Stream the complete 10,000-step chain

`evaluate_stream` accepts BEN, XBEN, and finalized BENDL input. It writes bounded,
Snappy-compressed Parquet batches and a versioned manifest. A missing output directory is
created. New score names are added to an existing run; `update=True` is only needed to replace
a name already stored there.

`samples` counts expanded chain steps. `accepted` counts encoded frames written to each table.
When consecutive assignments repeat, the Parquet `repetitions` column preserves their multiplicity.

In [ ]:
run_root = Path(tempfile.mkdtemp(prefix="gerrytools-scoring-"))
run_dir = run_root / "scores"
run = evaluator.evaluate_stream(bundle_path, run_dir, progress=True)
run.summary, run.metrics

In [ ]:
run.frames.head()

## Apply array formulas to streamed tallies

If you usually load a large DataFrame with `pd.read_parquet(...)`, think of
`run.read("district_totals")` as the equivalent operation for one scored metric. It eagerly
loads that metric and returns an ordinary pandas Series or DataFrame with meaningful index and
column labels. The underlying Parquet column names and table layout remain storage details.

A streamed chain can store a self-loop once with a repetition count instead of writing several
identical rows. By default, `run.read(...)` returns one row per accepted frame. Setting
`expand_repetitions=True` repeats those rows so the sample index covers all 10,000 original chain
steps in this example. This is convenient when downstream pandas code expects one unweighted row
per step, but the expanded result can use substantially more memory.

As with any large DataFrame load, an eager read must fit in memory. GerryTools warns when the
predicted peak reaches 2 GiB and raises `EvaluationMemoryError` at 8 GiB. If that happens, iterate
over `run.iter_batches(...)`; each batch has the same semantic pandas layout without loading the
whole metric at once. Use `allow_large=True` only when the machine deliberately has enough memory.
To keep the peak predictable, result reads decode Parquet columns serially, so very wide metrics
may load more slowly than a maximally parallel Parquet read.

After loading a metric, `.to_numpy()` provides the arrays used by `scoring.formulas`. For a typical
sample-by-district DataFrame, the array shape is `(samples, districts)`: formulas operate on every
leading sample or batch axis and treat the last axis as districts. Formulas that combine elections
expect an array shaped `(..., elections, districts)`.

In [ ]:
tallies = run.read("district_totals", expand_repetitions=True)
districts = tallies["total_pop_20"].columns

dem_2020 = tallies["pres_20_dem"].to_numpy()
rep_2020 = tallies["pres_20_rep"].to_numpy()
vote_shares_2020 = gs.formulas.district_vote_shares(dem_2020, rep_2020)
wins_2020 = gs.formulas.district_wins(dem_2020, rep_2020)

pd.concat(
    {
        "Democratic two-party share": pd.DataFrame(
            vote_shares_2020[:5],
            columns=districts,
        ),
        "Democratic win": pd.DataFrame(
            wins_2020[:5],
            columns=districts,
        ),
    },
    axis="columns",
).rename_axis(index="sample", columns=["quantity", "district"])

In [ ]:
partisan_scores_2020 = pd.DataFrame(
    {
        "seats": gs.formulas.seats(dem_2020, rep_2020),
        "overall_vote_share": gs.formulas.overall_vote_share(dem_2020, rep_2020),
        "efficiency_gap": gs.formulas.efficiency_gap(dem_2020, rep_2020),
        "simplified_efficiency_gap": gs.formulas.simplified_efficiency_gap(
            dem_2020,
            rep_2020,
        ),
        "mean_median": gs.formulas.mean_median(dem_2020, rep_2020),
        "partisan_bias_equal": gs.formulas.partisan_bias(
            dem_2020,
            rep_2020,
            turnout_model="equal",
        ),
        "partisan_bias_observed": gs.formulas.partisan_bias(
            dem_2020,
            rep_2020,
            turnout_model="observed",
        ),
        "partisan_gini_equal": gs.formulas.partisan_gini(
            dem_2020,
            rep_2020,
            turnout_model="equal",
        ),
        "partisan_gini_observed": gs.formulas.partisan_gini(
            dem_2020,
            rep_2020,
            turnout_model="observed",
        ),
    }
)
partisan_scores_2020.agg(["mean", "std", "min", "max"]).T

The fixture contains three presidential elections. Stacking them gives arrays with shape
`(plans, elections, districts)` for cross-election summaries.

In [ ]:
years = (2016, 2020, 2024)
dem_elections = np.stack(
    [tallies[f"pres_{year % 100:02d}_dem"].to_numpy() for year in years],
    axis=1,
)
rep_elections = np.stack(
    [tallies[f"pres_{year % 100:02d}_rep"].to_numpy() for year in years],
    axis=1,
)

wins_by_district = gs.formulas.party_wins_by_district(
    dem_elections,
    rep_elections,
)
pd.DataFrame(wins_by_district[:5], columns=districts)

In [ ]:
cross_election_scores = pd.DataFrame(
    {
        "competitive_contests": gs.formulas.competitive_contests(
            dem_elections,
            rep_elections,
            vote_share_margin=0.05,
        ),
        "swing_districts": gs.formulas.swing_districts(
            dem_elections,
            rep_elections,
        ),
        "democratic_districts": gs.formulas.party_districts(
            dem_elections,
            rep_elections,
        ),
        "republican_districts": gs.formulas.opposition_party_districts(
            dem_elections,
            rep_elections,
        ),
        "aggregate_democratic_seats": gs.formulas.aggregate_seats(
            dem_elections,
            rep_elections,
        ),
        "mean_signed_seat_vote_gap": gs.formulas.mean_signed_seat_vote_gap(
            dem_elections,
            rep_elections,
        ),
        "mean_absolute_seat_vote_gap": gs.formulas.mean_absolute_seat_vote_gap(
            dem_elections,
            rep_elections,
        ),
    }
)
cross_election_scores.agg(["mean", "std", "min", "max"]).T

## Derive population, demographic, and compactness scores

Population and demographic functions use district tallies from the same streamed table.
Schwartzberg compactness is derived from the scoring-engine Polsby-Popper output.

In [ ]:
population = tallies["total_pop_20"].to_numpy()
voting_age_population = tallies["total_vap_20"].to_numpy()
black_voting_age_population = tallies["bvap_20"].to_numpy()

population_deviations = gs.formulas.population_deviations(population)
bvap_share = gs.formulas.demographic_shares(
    black_voting_age_population,
    voting_age_population,
)
population_scores = pd.DataFrame(
    {
        "max_absolute_deviation": gs.formulas.max_absolute_population_deviation(
            population,
            relative_to_ideal=True,
        ),
        "maximum_deviation": gs.formulas.max_population_deviation(
            population,
            relative_to_ideal=True,
        ),
        "districts_above_40_percent_BVAP": gs.formulas.districts_above_threshold(
            black_voting_age_population,
            voting_age_population,
            threshold=0.4,
        ),
    }
)
population_scores.agg(["mean", "std", "min", "max"]).T

In [ ]:
polsby = run.read("polsby_popper", expand_repetitions=True).to_numpy()
schwartzberg = gs.formulas.schwartzberg(polsby)

pd.concat(
    {
        "population deviation": pd.DataFrame(
            population_deviations[:5],
            columns=districts,
        ),
        "BVAP share": pd.DataFrame(
            bvap_share[:5],
            columns=districts,
        ),
        "Schwartzberg": pd.DataFrame(
            schwartzberg[:5],
            columns=districts,
        ),
    },
    axis="columns",
)

## Inspect region-by-district tallies

`TallyByRegion` exposes values for each fixed region and proposed district. For one plan,
regions form the row index;
the first column level contains meaningful value names, and the second contains ordered district
labels.

In [ ]:
county_totals = single["county_totals"]
display(county_totals.head())

## Save and compare several runs

Give each assignment stream its own score directory. `evaluate_stream` creates the directory
and returns its open `EvaluationRun`. This tutorial has one BENDL fixture, so the example reuses
it for both paths; an actual comparison would pass a different BEN or BENDL file to each call.

In [ ]:
stats_dir = run_root / "stats"
run_1_path = stats_dir / "ben_file_1_scores"
run_2_path = stats_dir / "ben_file_2_scores"

totals = gs.PlanEvaluator(graph).add_metric(
    gs.Tally(
        "total_pop_20",
        "total_vap_20",
        result_name="district_totals",
    )
)
run_1_scores = totals.evaluate_stream(bundle_path, run_1_path)
run_2_scores = totals.evaluate_stream(bundle_path, run_2_path)
run_1_scores.summary, run_2_scores.summary

The returned object is already open. Use `read()` to extract a saved logical score; callers do
not need to find or combine the physical Parquet files themselves.

In [ ]:
run_1_totals = run_1_scores.read("district_totals")
run_2_totals = run_2_scores.read("district_totals")
run_1_totals.head()

A new result name is added without a flag. Reopen the run afterward so the object sees the new
manifest, then read both the old totals and the new cut-edge score.

In [ ]:
edges = gs.PlanEvaluator(graph).add_metric(gs.CutEdges())
edges.evaluate_stream(bundle_path, run_1_path)
edges.evaluate_stream(bundle_path, run_2_path)

run_1_scores = gs.EvaluationRun.open(run_1_path)
run_2_scores = gs.EvaluationRun.open(run_2_path)
old_totals = run_1_scores.read("district_totals")
new_cut_edges = run_1_scores.read("cut_edges")
display(old_totals.head())
new_cut_edges.head()

The directory name is included in every filename, so a copied Parquet file still identifies
its run:

```text
stats/
├── ben_file_1_scores/
│   ├── manifest__ben_file_1_scores.json
│   ├── cut_edges__ben_file_1_scores.parquet
│   └── district_totals/
│       ├── total_pop_20_tallies__ben_file_1_scores.parquet
│       └── total_vap_20_tallies__ben_file_1_scores.parquet
└── ben_file_2_scores/
    ├── manifest__ben_file_2_scores.json
    ├── cut_edges__ben_file_2_scores.parquet
    └── district_totals/
        ├── total_pop_20_tallies__ben_file_2_scores.parquet
        └── total_vap_20_tallies__ben_file_2_scores.parquet
```

Use `update=True` only when replacing a result with the same name. Scores not registered on
the replacement evaluator stay as they are.

In [ ]:
replacement = gs.PlanEvaluator(graph).add_metric(
    gs.Tally("total_pop_20", result_name="district_totals")
)
replacement.evaluate_stream(bundle_path, run_1_path, update=True)

run_1_scores = gs.EvaluationRun.open(run_1_path)
updated_totals = run_1_scores.read("district_totals")
preserved_cut_edges = run_1_scores.read("cut_edges")
updated_totals.head(), preserved_cut_edges.head()

`evaluate_stream` checks that the sample count, accepted-frame count, and district IDs fit the
existing directory. You are responsible for passing the same assignment stream when adding or
replacing scores. Reopen an `EvaluationRun` after changing its directory; an object opened
earlier keeps the old manifest in memory.

## Result contracts

`evaluate` returns `PlanEvalResult`; indexing it by metric name returns a scalar, Series, or
DataFrame with semantic labels. `evaluate_many` returns `EnsembleEvalResult`; its first axis uses
`sample_ids` when supplied. Region results use `(sample, region)` rows for ensembles and region
rows for one plan, with `(metric, district)` columns in both cases. `array(name)` is available
when a canonical immutable NumPy view is preferable.

`evaluate_stream` returns `EvaluationRun` after writing Parquet score tables plus a JSON
manifest. Tallies and region tallies use one table per requested attribute; other metrics use
one table. Every filename ends with the run-directory name. `frames` exposes accepted-frame
offsets and repetition counts. `read()` restores
the same logical pandas shapes at either accepted-frame or expanded-sample resolution, while
`raw()` remains available for physical Parquet access. Large results can instead be processed
with `iter_batches()`, `iter_raw_batches()`, or `iter_frame_batches()` without materializing a
whole table. Array formulas deliberately remain
separate, so they work with in-memory results, streamed tables, or arrays produced elsewhere.

Sequence assignments must follow graph-node order. Mapping assignments are aligned by node
identifier; both forms are captured by the exported `gerrytools.scoring.Assignment` alias.
Geometry must be projected for area and distance metrics. Consult each metric or
formula docstring for its formula, tie convention, turnout model, and literature
references.